# 02 — Data Preprocessing

**AttriSense · Employee Attrition Prediction & Analytics System**

---

## Introduction

This notebook implements the full preprocessing pipeline for the IBM HR Analytics Employee Attrition dataset. Every transformation is explicit, justified, and logged — nothing is applied silently.

**Scope of this notebook:**

| Step | Action |
|------|--------|
| Feature inspection | Review all 35 raw columns individually |
| Duplicate handling | Detect and remove duplicate rows/IDs |
| Missing values | Inspect and document (impute only if needed) |
| Column removal | Drop only columns with proven zero variance |
| Outlier analysis | IQR diagnostic; capping only where justified |
| Encoding | One-hot for nominal; integer for ordinal |
| Scaling | Applied only when required (off by default for tree models) |
| Persistence | Save cleaned + preprocessed datasets and transformer |

**Prerequisite:** `01_Data_Understanding.ipynb`

**Outputs:**
- `data/processed/employee_attrition_cleaned.parquet`
- `data/processed/employee_attrition_preprocessed.parquet`
- `models/feature_transformer.joblib`

## Library Imports

Reusable logic lives in `src/attrisense/data/preprocessing.py`. The notebook calls those functions and displays every intermediate result.

In [ ]:
import pandas as pd

from attrisense.config import load_config
from attrisense.data import (
    feature_registry_dataframe,
    handle_duplicates,
    inspect_missing_values,
    inspect_outliers_iqr,
    load_raw_data,
    run_preprocessing_pipeline,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.2f}".format)

config = load_config()
df = load_raw_data()

print(f"Loaded raw data: {df.shape[0]:,} rows × {df.shape[1]} columns")

## Step 1 — Inspect Every Feature

Each of the 35 raw columns is reviewed below. The `action` and `justification` columns document the preprocessing decision before any code runs.

**Decision categories:**
- **Keep** — column enters the feature set as-is
- **Keep as integer** — ordinal scale with natural order
- **Keep; one-hot encode** — nominal category without order
- **Keep; encode to 0/1** — target variable
- **Keep in cleaned data; exclude from feature matrix** — identifier
- **Remove** — zero variance or no predictive value

In [ ]:
registry = feature_registry_dataframe()
registry

### Verify registry against actual data

Cross-check documented decisions with observed unique values and dtypes from the loaded dataset.

In [ ]:
verification = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "unique_values": [df[c].nunique() for c in df.columns],
    "sample_values": [
        str(sorted(df[c].unique()[:5])) if df[c].nunique() <= 10
        else f"[{df[c].min()}, ..., {df[c].max()}]"
        if pd.api.types.is_numeric_dtype(df[c])
        else str(df[c].value_counts().head(2).to_dict())
        for c in df.columns
    ],
})
verification

## Step 2 — Handle Duplicate Rows

Duplicate records would inflate sample counts and could leak across train/test splits. We check both full-row duplicates and repeated `EmployeeNumber` values.

**Expected result (from data understanding):** 0 duplicates. The handler still runs so the pipeline is safe if the source file changes.

In [ ]:
deduped, dup_report = handle_duplicates(df, config.id_column)

print(f"Duplicate rows before     : {dup_report.duplicate_rows}")
print(f"Duplicate EmployeeNumber  : {dup_report.duplicate_ids}")
print(f"Rows removed              : {dup_report.rows_removed}")
print(f"Shape after deduplication : {deduped.shape}")
print(f"Action                    : {dup_report.action_taken}")

## Step 3 — Inspect Missing Values

Missing data requires an imputation strategy. We count nulls per column before deciding whether to impute, drop rows, or drop columns.

**Decision rule:** Impute only when missing values exist. This dataset was verified in the data understanding notebook to have complete records.

In [ ]:
missing_report = inspect_missing_values(deduped)

print(f"Total missing values : {missing_report.total_missing}")
print(f"Columns affected     : {len(missing_report.columns_with_missing)}")
print(f"Action               : {missing_report.action_taken}")

if missing_report.columns_with_missing:
    print(pd.Series(missing_report.columns_with_missing, name="missing_count"))
else:
    print("No imputation applied.")

## Step 4 — Remove Irrelevant Columns

Columns are removed **only** when they carry zero variance. Each drop is justified individually — we do not remove columns based on correlation or feature importance at this stage.

| Column | Unique Values | Reason for Removal |
|--------|---------------|-------------------|
| `EmployeeCount` | 1 (always 1) | No variance — every row is a single employee |
| `Over18` | 1 (always "Y") | No variance — all employees are adults |
| `StandardHours` | 1 (always 80) | No variance — identical for all records |

**Not removed:**
- `EmployeeNumber` — unique identifier, kept for traceability but excluded from the model feature matrix
- All other columns — retained for encoding and modeling

In [ ]:
constant_cols = [c for c in deduped.columns if deduped[c].nunique() == 1]

constant_detail = pd.DataFrame({
    "column": constant_cols,
    "sole_value": [deduped[c].iloc[0] for c in constant_cols],
    "in_config_drop_list": [c in config.drop_columns for c in constant_cols],
})
print("Constant columns detected in raw data:")
constant_detail

## Step 5 — Outlier Inspection

We apply the IQR rule (Q1 − 1.5×IQR, Q3 + 1.5×IQR) as a **diagnostic** tool. Flagged values are not automatically removed or capped.

**Why no automatic capping:**
- `MonthlyIncome` — high earners are valid, not errors
- `TotalWorkingYears` / `YearsAtCompany` — senior employees with 30–40 years tenure are real
- `PerformanceRating` — only two values (3, 4); IQR flags most rows as "outliers" because IQR = 0
- `TrainingTimesLastYear` — count variable 0–6; higher counts are valid
- `StockOptionLevel` — level 3 is the maximum benefit tier, not an anomaly

`cap_outliers` is set to `false` in `configs/config.yaml`.

In [ ]:
outlier_cols = config.features.continuous + config.features.ordinal
outlier_report = inspect_outliers_iqr(
    deduped,
    outlier_cols,
    multiplier=config.preprocessing.outlier_iqr_multiplier,
)

print(outlier_report.action_taken)
outlier_report.details.sort_values("outlier_count", ascending=False)

## Step 6 — Encode Categorical Variables

Encoding strategy depends on whether a column has a natural order.

### Nominal → One-Hot Encoding (`drop='first'`)

| Column | Levels | Rationale |
|--------|--------|-----------|
| `BusinessTravel` | 3 | No order between travel categories |
| `Department` | 3 | Departments are unordered |
| `EducationField` | 6 | Fields of study have no ranking |
| `Gender` | 2 | Unordered categories |
| `JobRole` | 9 | Job titles are not ordinal |
| `MaritalStatus` | 3 | Unordered categories |
| `OverTime` | 2 | Yes/No flag |

Using `drop='first'` avoids the dummy variable trap (multicollinearity).

### Ordinal → Keep as Integer

| Column | Range | Rationale |
|--------|-------|-----------|
| `Education` | 1–5 | Below College → Doctor |
| `EnvironmentSatisfaction` | 1–4 | Likert scale |
| `JobInvolvement` | 1–4 | Likert scale |
| `JobLevel` | 1–5 | Hierarchy level |
| `JobSatisfaction` | 1–4 | Likert scale |
| `PerformanceRating` | 3–4 | Rating scale (limited levels in data) |
| `RelationshipSatisfaction` | 1–4 | Likert scale |
| `StockOptionLevel` | 0–3 | Benefit tier |
| `WorkLifeBalance` | 1–4 | Likert scale |

One-hot encoding ordinal columns would discard the ordering information.

### Target → Binary Integer

| Original | Encoded |
|----------|---------|
| `Yes` | 1 |
| `No` | 0 |

In [ ]:
print("Nominal columns (one-hot):")
for col in config.features.nominal:
    print(f"  {col}: {deduped[col].nunique()} levels → {deduped[col].unique().tolist()}")

print("\nOrdinal columns (kept as integer):")
for col in config.features.ordinal:
    print(f"  {col}: {sorted(deduped[col].unique().tolist())}")

print("\nContinuous columns (kept as numeric):")
print(f"  {config.features.continuous}")

## Step 7 — Scaling Decision

`StandardScaler` (zero mean, unit variance) is needed for distance-based and linear models (Logistic Regression, SVM) but **not** for tree-based models (Random Forest, XGBoost).

| Setting | Value | Reason |
|---------|-------|--------|
| `apply_scaling` | `False` | Default pipeline targets XGBoost; trees are scale-invariant |
| `scale_when_required` | 14 continuous columns | Available when linear models are trained later |

The fitted `ColumnTransformer` is saved to `models/feature_transformer.joblib`. A separate modeling notebook can re-fit with `apply_scaling=True` for linear models.

In [ ]:
APPLY_SCALING = False

print(f"Scaling applied in this run: {APPLY_SCALING}")
print(f"Columns configured for scaling when required:")
for col in config.features.scale_when_required:
    print(f"  - {col}")

## Step 8 — Run Pipeline and Save Outputs

The full pipeline executes all steps above in sequence and writes three artifacts to disk.

In [ ]:
cleaned, preprocessed, report = run_preprocessing_pipeline(
    df,
    config=config,
    apply_scaling=APPLY_SCALING,
    save_artifacts=True,
)

print("=" * 55)
print("PREPROCESSING REPORT")
print("=" * 55)
print(f"Input shape        : {report.input_shape}")
print(f"Cleaned shape      : {report.cleaned_shape}")
print(f"Preprocessed shape : {report.preprocessed_shape}")
print(f"Dropped columns    : {report.dropped_columns}")
print(f"Duplicate action   : {report.duplicate_report.action_taken}")
print(f"Missing action     : {report.missing_report.action_taken}")
print(f"Outlier action     : {report.outlier_report.action_taken}")
print(f"Scaling applied    : {report.scaling_applied}")
print(f"Feature columns    : {report.encoding_summary['output_feature_count']}")
print()
print("Saved artifacts:")
for name, path in report.output_paths.items():
    print(f"  {name}: {path}")

### Cleaned dataset preview

Human-readable dtypes preserved — suitable for EDA and reporting.

In [ ]:
cleaned.head(3)

### Preprocessed dataset preview

Model-ready matrix: identifier, target (string + label), and encoded features.

In [ ]:
feature_cols = [c for c in preprocessed.columns if c not in (
    config.id_column, config.target_column, "Attrition_label"
)]
print(f"Model feature columns ({len(feature_cols)}):")
print(feature_cols)

preprocessed.head(3)

## Step 9 — Preprocessing Summary

All decisions applied in this notebook:

In [ ]:
summary = pd.DataFrame([
    {"step": "Duplicates", "decision": report.duplicate_report.action_taken},
    {"step": "Missing values", "decision": report.missing_report.action_taken},
    {
        "step": "Column removal",
        "decision": f"Dropped {report.dropped_columns} — zero variance only",
    },
    {"step": "Outliers", "decision": report.outlier_report.action_taken},
    {
        "step": "Nominal encoding",
        "decision": "OneHotEncoder(drop='first') on 7 columns",
    },
    {
        "step": "Ordinal encoding",
        "decision": "Kept as integer on 9 columns",
    },
    {
        "step": "Target encoding",
        "decision": "Yes→1, No→0 stored as Attrition_label",
    },
    {
        "step": "Scaling",
        "decision": (
            "StandardScaler applied to 14 continuous columns"
            if report.scaling_applied
            else "Not applied (tree-model default; scaler config ready for linear models)"
        ),
    },
    {
        "step": "Saved cleaned data",
        "decision": report.output_paths.get("cleaned", "N/A"),
    },
    {
        "step": "Saved preprocessed data",
        "decision": report.output_paths.get("preprocessed", "N/A"),
    },
    {
        "step": "Saved transformer",
        "decision": report.output_paths.get("transformer", "N/A"),
    },
])
summary

**Next step:** `03_EDA.ipynb` — exploratory analysis on the cleaned dataset, or proceed to feature engineering / model building using `employee_attrition_preprocessed.parquet`.